In [19]:
import pandas as pd
import numpy as np

import re
import string

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vishw\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vishw\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\vishw\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [47]:
df = pd.read_excel("../dataset/training_dataset.xlsx")

df.head()

,City,Restaurant,Reviewer,Rating,Review,Sentiment,review_date,date,aspect,emotion,intent,severity
0,Indore,The Village By Maa Ki Rasoi Sudama Nagar,Monika Jajoo,5.0,All the things are good,Positive,Not Available,05-07-2026,General,Happy,Appreciation,Low
1,Indore,The Village By Maa Ki Rasoi Sudama Nagar,Amar Goswami,5.0,Nice food and service,Positive,Not Available,18-06-2026,Food,Happy,Appreciation,Low
2,Indore,The Village By Maa Ki Rasoi Sudama Nagar,Vishal Sharma,5.0,Amazing over all nice.,Positive,Not Available,02-07-2026,General,Happy,Appreciation,Low
3,Indore,The Village By Maa Ki Rasoi Sudama Nagar,Shoeb Inamdar,1.0,test is not good,Negative,Not Available,22-06-2026,General,Angry,Complaint,Critical
4,Indore,The Village By Maa Ki Rasoi Sudama Nagar,Venkatasai Palla,5.0,"Food was superrrr , I came from Andhra Pradesh...",Positive,Not Available,01-07-2026,Food,Happy,Appreciation,Low


In [48]:
df.shape

(4285, 12)

In [49]:
df.columns

Index(['City', 'Restaurant', 'Reviewer', 'Rating', 'Review', 'Sentiment',
       'review_date', 'date', 'aspect', 'emotion', 'intent', 'severity'],
      dtype='object')

In [50]:
df = df[[
    "Restaurant",
    "Reviewer",
    "Rating",
    "Review",
    "Sentiment"
]]

In [51]:
df.columns = [
    "restaurant_name",
    "reviewer_name",
    "rating",
    "review_text",
    "sentiment"
]

In [52]:
df.isnull().sum()

restaurant_name     0
reviewer_name       0
rating              0
review_text        86
sentiment           0
dtype: int64

In [53]:
df = df.dropna()

In [54]:
df = df.drop_duplicates(
    subset=["restaurant_name", "review_text"]
)

df.shape

(4199, 5)

In [55]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

In [56]:
df["clean_review"] = df["review_text"].apply(clean_text)

df[["review_text", "clean_review"]].head()

,review_text,clean_review
0,All the things are good,thing good
1,Nice food and service,nice food service
2,Amazing over all nice.,amazing nice
3,test is not good,test good
4,"Food was superrrr , I came from Andhra Pradesh...",food superrrr came andhra pradesh food aswm be...


In [57]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df["sentiment"])

df[["sentiment", "label"]].head()

,sentiment,label
0,Positive,2
1,Positive,2
2,Positive,2
3,Negative,0
4,Positive,2


In [58]:
df["sentiment"].value_counts()

sentiment
Positive    3335
Negative     632
Neutral      232
Name: count, dtype: int64

In [59]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["clean_review"])

y = df["label"]

print(X.shape)

(4199, 5000)


In [61]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(3359, 5000)
(840, 5000)


In [62]:
X_train = X_train.toarray()
X_test = X_test.toarray()

print(X_train.shape)
print(X_test.shape)

(3359, 5000)
(840, 5000)


In [63]:
import tensorflow as tf

print(tf.__version__)

2.20.0


In [64]:
model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(X_train.shape[1],)),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(64, activation="relu"),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(32, activation="relu"),

    tf.keras.layers.Dense(3, activation="softmax")

])

In [65]:
model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 650,563 (2.48 MB)

 Trainable params: 650,563 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

In [66]:
early_stop = tf.keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True

)

In [67]:
history = model.fit(

    X_train,

    y_train,

    validation_split=0.2,

    epochs=20,

    batch_size=32,

    callbacks=[early_stop],

    verbose=1

)

Epoch 1/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7812 - loss: 0.6881 - val_accuracy: 0.7917 - val_loss: 0.4800
Epoch 2/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8359 - loss: 0.3687 - val_accuracy: 0.8542 - val_loss: 0.3764
Epoch 3/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9125 - loss: 0.2397 - val_accuracy: 0.8705 - val_loss: 0.3766
Epoch 4/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9289 - loss: 0.1684 - val_accuracy: 0.8765 - val_loss: 0.4134
Epoch 5/20
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9405 - loss: 0.1350 - val_accuracy: 0.8810 - val_loss: 0.4174


In [68]:
loss, accuracy = model.evaluate(

    X_test,

    y_test

)

print("Loss :", loss)

print("Accuracy :", accuracy)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8655 - loss: 0.3664  
Loss : 0.3663972318172455
Accuracy : 0.8654761910438538


In [69]:
predictions = model.predict(X_test)

predicted_labels = predictions.argmax(axis=1)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [70]:
from sklearn.metrics import classification_report

print(

    classification_report(

        y_test,

        predicted_labels

    )

)

              precision    recall  f1-score   support

           0       0.63      0.72      0.67       127
           1       0.00      0.00      0.00        46
           2       0.92      0.95      0.93       667

    accuracy                           0.87       840
   macro avg       0.51      0.56      0.53       840
weighted avg       0.82      0.87      0.84       840



c:\Users\vishw\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\vishw\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\vishw\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [71]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(

    y_test,

    predicted_labels

)

print(cm)

[[ 91   0  36]
 [ 23   0  23]
 [ 31   0 636]]


In [72]:
model.save("../models/sentiment_model.keras")

In [74]:
import joblib

joblib.dump(

    vectorizer,

    "../models/tfidf_vectorizer.pkl"

)

['../models/tfidf_vectorizer.pkl']

In [75]:
joblib.dump(

    label_encoder,

    "../models/label_encoder.pkl"

)

['../models/label_encoder.pkl']